In [ ]:
from psycopg2.errors import ProgrammingError
from psycopg2 import connect
from dotenv import load_dotenv
from os import getenv
from pprint import pprint

load_dotenv()

In [2]:
def run_query(command: str) -> None:
    db_user = getenv("DB_USER")
    db_password = getenv("DB_PASSWORD")
    db_name = getenv("DB_NAME")
    db_host = getenv("DB_HOST")
  
    connector = connect(
        dbname=db_name,
        user=db_user,
        password=db_password,
        host=db_host,
    )

    connector.autocommit = True
    cursor = connector.cursor()


    cursor.execute(command)
    try:
      result = cursor.fetchall()
    except ProgrammingError as e:
      cursor.close()
      connector.close()
      return None

    cursor.close()
    connector.close()

    if result == None:
       return None

    pprint(result)

Query 1: Gesamte Bewerbungsübersicht
Welche Studenten haben sich in welcher Bewerbungsrunde beworben, mit welchem Status, welcher Wunschuniversität,
welcher Priorität, welcher Auswahlentscheidung und welchem Nominierungsstatus?

In [3]:
query:str = """
select
    *
from
    public.v_bewerbungsuebersicht vb;
"""

result = run_query(query)

OperationalError: connection to server on socket "/var/run/postgresql/.s.PGSQL.5432" failed: FATAL:  database "jonathan" does not exist


Query 2: Angenommene Bewerbungen
Welche Bewerbungen wurden angenommen und für welche Partneruniversitäten waren sie vorgesehen?


In [31]:
query:str = """
select
    vb.bewerbungsstatus,
    vb.bewerbung_id,
    vb.vorname,
    vb.nachname,
    vb.bewerbungsrunde,
    vb.partneruniversitaet,
    vb.nominierungsstatus,
    vb.auswahlstatus,
    vb.prioritaet
from
    public.v_bewerbungsuebersicht vb
where
    vb.bewerbungsstatus = 'angenommen';
"""

result = run_query(query)

[('angenommen',
  0,
  'Marie',
  'Johanna',
  'WiSe 2026',
  'Université Paris Cité',
  'nominiert',
  'angenommen',
  1),
 ('angenommen',
  0,
  'Marie',
  'Johanna',
  'WiSe 2026',
  'Université Jean Moulin Lyon 3',
  'nominiert',
  'angenommen',
  2)]


Query 3: Bewerbungen nach Status zählen
Wie viele Bewerbungen gibt es pro Status?

In [32]:
query:str = """
select
    b.status,
    COUNT(b.bewerbung_id)
from
    public.bewerbung b
group by
    b.status;
"""

result = run_query(query)

[('eingereicht', 1), ('abgelehnt', 1), ('angenommen', 1)]


Query 4: Nachfrage pro Partneruniversität
Wie viele Bewerbungspräferenzen gibt es pro Partneruniversität?

In [33]:
query:str = """
select
    COUNT(*) AS nachfrage,
    vb.partneruniversitaet
from
    public.v_bewerbungsuebersicht vb
group by
    vb.partneruniversitaet;
"""

result = run_query(query)

[(1, 'Université Jean Moulin Lyon 3'),
 (1, 'Stockholms universitet'),
 (1, 'Université Paris Cité'),
 (2, 'Universiteit van Amsterdam'),
 (1, 'Universidad Complutense de Madrid')]


Query 5: Sprachnachweise pro Bewerbung
Welche Bewerbung verwendet welche Sprachnachweise?

In [34]:
query:str = """
select
    b.bewerbung_id,
    b.matrikelnummer,
    s.vorname,
    s.nachname,
    s2.sprachnachweis_id,
    s2.sprache,
    s2.niveau,
    s2.nachweistyp
from
    public.bewerbung b
inner join public.student s on
    b.matrikelnummer = s.matrikelnummer
inner join public.bewerbung_sprachnachweis bs on
    b.bewerbung_id = bs.bewerbung_id
inner join public.sprachnachweis s2 on
    s.matrikelnummer = s2.matrikelnummer
where
    bs.sprachnachweis_id = s2.sprachnachweis_id;
"""

result = run_query(query)

[(0, 618492, 'Marie', 'Johanna', 0, 'Französisch', 'B2', 'DELF'),
 (0, 618492, 'Marie', 'Johanna', 1, 'Englisch', 'C1', 'Cambridge Certificate'),
 (1, 629048, 'Niko', 'Thin', 2, 'Englisch', 'C1', 'TOEFL'),
 (2, 658392, 'Max', 'Mustermensch', 3, 'Englisch', 'B2', 'IELTS')]


Query 6: Auslandsaufenthalt mit Nominierung und Partneruniversität
Welcher nominierte Student hat welchen Auslandsaufenthalt an welcher Partneruniversität?

In [35]:
query:str = """
SELECT
    p.name AS partneruniversitaet,
    p.partneruniversitaet_erasmus_code,
    p.stadt,
    a.semester,
    a.status AS aufenthalt_status,
    a.startdatum,
    a.enddatum,
    n.status AS nominierung_status,
    ak.studienphase,
    s.vorname,
    s.nachname
FROM public.auslandsaufenthalt a
INNER JOIN public.nominierung n
    ON a.nominierung_id = n.nominierung_id
INNER JOIN public.bewerbung b
    ON n.bewerbung_id = b.bewerbung_id
INNER JOIN public.student s
    ON b.matrikelnummer = s.matrikelnummer
INNER JOIN public.austauschkontingent ak
    ON n.austauschkontingent_id = ak.austauschkontingent_id
INNER JOIN public.partneruniversitaet p
    ON ak.partneruniversitaet_erasmus_code = p.partneruniversitaet_erasmus_code
WHERE n.status = 'nominiert';
"""

result = run_query(query)

[('Université Paris Cité',
  'F PARIS001',
  'Paris',
  'WiSe',
  'abgeschlossen',
  datetime.date(2026, 9, 15),
  datetime.date(2027, 2, 28),
  'nominiert',
  'Bachelor',
  'Marie',
  'Johanna')]


Query 7: Learning Agreement mit Versionen
Welche Versionen gibt es zu einem Learning Agreement und warum wurden sie erstellt?

In [36]:
query:str = """
select
    lav.versionsnummer,
    la.learning_agreement_id,
    lav.grund,
    la.erstellungsdatum AS la_erstellungsdatum,
    lav.status AS versionsstatus,
    lav.erstellungsdatum AS versions_erstellungsdatum,
    a.auslandsaufenthalt_id,
    a.semester,
    a.status AS aufenthalt_status
from
    public.learning_agreement la
inner join public.learning_agreement_version lav on
    la.learning_agreement_id = lav.learning_agreement_id
inner join public.auslandsaufenthalt a on
    la.auslandsaufenthalt_id = a.auslandsaufenthalt_id;
"""

result = run_query(query)

[(2,
  0,
  'Aenderung nach Kursueberschneidung an der Partneruniversitaet',
  datetime.date(2026, 7, 5),
  'genehmigt',
  datetime.date(2026, 10, 20),
  0,
  'WiSe',
  'abgeschlossen'),
 (1,
  0,
  'Erste Kursplanung vor Beginn des Auslandsaufenthalts',
  datetime.date(2026, 7, 5),
  'genehmigt',
  datetime.date(2026, 7, 5),
  0,
  'WiSe',
  'abgeschlossen')]


Query 8: Kurszuordnungen im Learning Agreement
Welche Gastkurse wurden welchen HU-Modulen zugeordnet und welchen Status haben diese Zuordnungen?

In [37]:
query:str = """
select
    g."name" AS gastkurs,
    hm."name" AS hu_modul,
    hm.fach,
    hm.ects,
    g.fach,
    g.ects,
    g.partneruniversitaet_erasmus_code,
    lav.learning_agreement_id,
    k.learning_agreement_version_id,
    k.status AS zuordnungsstatus
from
    public.kurszuordnung k
inner join public.learning_agreement_version lav on
    k.learning_agreement_version_id = lav.learning_agreement_version_id
inner join public.gastkurs g on
    k.gastkurs_id = g.gastkurs_id
inner join public.hu_modul hm on
    k.hu_modul_id = hm.hu_modul_id;
"""

result = run_query(query)

[('General Chemistry',
  'Allgemeine Chemie',
  'Chemie',
  6,
  'Chemie',
  6,
  'F PARIS001',
  0,
  0,
  'genehmigt'),
 ('Organic Chemistry',
  'Organische Chemie',
  'Chemie',
  6,
  'Chemie',
  6,
  'F PARIS001',
  0,
  0,
  'genehmigt'),
 ('French Language Course',
  'Ueberfachlicher Wahlpflichtbereich',
  'UeWP',
  5,
  'Sprache',
  3,
  'F PARIS001',
  0,
  0,
  'genehmigt'),
 ('General Chemistry',
  'Allgemeine Chemie',
  'Chemie',
  6,
  'Chemie',
  6,
  'F PARIS001',
  0,
  1,
  'genehmigt'),
 ('Organic Chemistry',
  'Organische Chemie',
  'Chemie',
  6,
  'Chemie',
  6,
  'F PARIS001',
  0,
  1,
  'genehmigt'),
 ('Laboratory Methods in Chemistry',
  'Chemisches Praktikum',
  'Chemie',
  6,
  'Chemie',
  6,
  'F PARIS001',
  0,
  1,
  'genehmigt'),
 ('French Language Course',
  'Ueberfachlicher Wahlpflichtbereich',
  'UeWP',
  5,
  'Sprache',
  3,
  'F PARIS001',
  0,
  1,
  'geaendert')]


Query 9: Summe anerkannter ECTS pro Aufenthalt
Wie viele ECTS wurden für einen Auslandsaufenthalt insgesamt anerkannt?


In [38]:
query:str = """
select
    a.auslandsaufenthalt_id,
    COUNT(a2.anerkannte_ects) AS anerkannte_ects
from
    public.anerkennungsantrag a
inner join public.anerkennungsentscheidung a2 on
    a.anerkennungsantrag_id = a2.anerkennungsantrag_id
inner join public.auslandsaufenthalt a3 on
    a.auslandsaufenthalt_id = a3.auslandsaufenthalt_id
group by
    a.auslandsaufenthalt_id;
"""

result = run_query(query)

[(0, 4)]


Query 10: Teilweise oder nicht anerkannte Leistungen
Welche erbrachten Leistungen wurden nur teilweise anerkannt oder abgelehnt?


In [39]:
query:str = """
select
    hm.ects,
    hm."name",
    a.hu_modul_id,
    a.anerkannte_ects,
    a.entscheidung,
    el.gastkurs_id,
    el.ects,
    el.note,
    el.bestanden
from
    public.erbrachte_leistung el
inner join public.anerkennungsentscheidung a on
    el.erbrachteleistungs_id = a.erbrachteleistungs_id
inner join public.hu_modul hm on
    a.hu_modul_id = hm.hu_modul_id
WHERE a.entscheidung IN ('teilweise_anerkannt', 'abgelehnt');
"""

result = run_query(query)

[(5,
  'Ueberfachlicher Wahlpflichtbereich',
  2,
  3,
  'teilweise_anerkannt',
  2,
  3,
  Decimal('1.3'),
  True)]
